In [ ]:
from src.transformer import Transformer
import torch.nn.functional as F
import src.utils as utils
import numpy as np
import torch

test_str = '我见青山多妩媚，料青山见我应如是。'

checkpoint = 'results/checkpoint_best.pt'
state_dict = torch.load(checkpoint, map_location=utils.device)

model_args = state_dict['model_args']
model_kwargs = state_dict['model_kwargs']
model = Transformer(**model_args, **model_kwargs)
model.eval()
model.load_state_dict(state_dict['model'])

src_dict = model.args['src_dict']
tgt_dict = model.args['tgt_dict']

In [ ]:
beam_width = 10

src_tokens = src_dict.string_to_tokenIds(test_str)
go_slice   = torch.tensor(tgt_dict.sos_id)

src_tokens = src_tokens.repeat((beam_width, 1)) # [beam_width, sent_len]
go_slice   = go_slice.repeat((beam_width, 1))   # [beam_width, 1]

prev_words = go_slice
next_words = None
max_len = 200

for curr_time_step in range(max_len):
    with torch.no_grad():
        # [beam_width, 1, vocab_size]
        output = model(src_tokens, prev_words)
        output = F.log_softmax(output, dim=-1)
    
    if curr_time_step == 0:
        _, next_candidates = torch.topk(output, beam_width, dim=-1)
        next_candidates = next_candidates[0]
    else:
        # [beam_width, time_step, vocab_size] ---> [time_step, beam_width*vocab_size]
        output = output.transpose(0, 1).reshape(-1, beam_width*len(tgt_dict))
        # sum over log probability
        output = torch.sum(output, dim=0)
        if curr_time_step == max_len-1:
            _, next_candidates = torch.topk(output, 1, dim=-1)
        else:
            _, next_candidates = torch.topk(output, beam_width, dim=-1)
        # shift back indices that are larger than vocab_size
        i=0
        while True:
            i+=1
            shiftback_mask = next_candidates >= len(tgt_dict)
            shiftback = torch.zeros_like(next_candidates)
            shiftback = shiftback.masked_fill(shiftback_mask, len(tgt_dict))
            shiftback = shiftback.masked_fill(~shiftback_mask, 0)
            next_candidates -= shiftback
            if (next_candidates < len(tgt_dict)).all():
                if curr_time_step == max_len-1:
                    next_candidates = torch.cat([next_words[i], next_candidates])
                    break
                next_candidates = next_candidates.unsqueeze(-1)
                next_candidates = torch.cat([next_words, next_candidates], dim=1)
                break
    
    if curr_time_step == max_len-1:
        next_words = next_candidates
    else:
        next_words = next_candidates.view(beam_width, -1)
        prev_words = torch.cat([go_slice, next_words], dim=1)

In [ ]:
# segment into sentences
output_sent = next_words

# remove padding
first_eos = np.where(output_sent == tgt_dict.eos_id)[0]
if len(first_eos) > 0:
    output_sent = output_sent[:first_eos[0]]

# convert arrays of indices into strings of words
output_sent = tgt_dict.tokenIds_to_string(output_sent)
print(output_sent)